# Used Car Resale Dataset — Data Preprocessing

Download the Used Car Resale CSV from the LMS **Study Material** tab, then run the single code cell and upload it. The workflow uses train-only fitting to prevent data leakage.

In [ ]:
# Leakage-safe Used Car Resale preprocessing — single Google Colab cell
import pandas as pd
import numpy as np
from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Upload the Used Car Resale CSV from the LMS.
uploaded = files.upload()
csv_files = [name for name in uploaded if name.lower().endswith('.csv')]
if not csv_files:
    raise FileNotFoundError('Please upload the Used Car Resale CSV file.')

df = pd.read_csv(csv_files[0])
df.columns = df.columns.str.strip().str.replace(r'\s+', '_', regex=True)
pd.set_option('display.max_columns', None)

def find_column(candidates):
    normalized = {col.lower().replace('_', ' ').replace('-', ' ').strip(): col for col in df.columns}
    for candidate in candidates:
        if candidate in normalized:
            return normalized[candidate]
    for name, original in normalized.items():
        if any(candidate in name for candidate in candidates):
            return original
    return None

target = find_column(['selling price', 'resale price', 'price', 'target'])
if target is None:
    raise ValueError('Target column not found. Rename the resale-price field to Selling_Price, Resale_Price, or Price.')

# Convert typical car numeric fields such as '23.4 kmpl' or '1248 CC' to numbers.
numeric_keywords = ['year', 'km', 'mileage', 'engine', 'power', 'seats', 'price', 'age']
for column in df.columns:
    if column == target or any(key in column.lower() for key in numeric_keywords):
        converted = pd.to_numeric(df[column].astype(str).str.replace(r'[^0-9.-]', '', regex=True), errors='coerce')
        if converted.notna().sum() >= max(1, df[column].notna().sum() * 0.5):
            df[column] = converted

# Remove rows with no target value; a predictive model cannot train on them.
before_target_drop = len(df)
df = df.dropna(subset=[target]).drop_duplicates().copy()
print('=' * 90)
print(f'USED CAR RESALE PREPROCESSING: {csv_files[0]}')
print('=' * 90)
print(f'Input shape: {before_target_drop:,} rows × {len(df.columns)} columns')
print(f'Rows retained after removing missing targets and duplicates: {len(df):,}')
print(f'Target variable: {target}')
display(df.head())

# Separate features and target before splitting. Identifiers are excluded from modelling features.
id_columns = [col for col in df.columns if col.lower() in ['id', 'car_id', 'unnamed:_0'] or col.lower().endswith('_id')]
feature_columns = [col for col in df.columns if col not in [target] + id_columns]
X = df[feature_columns].copy()
y = df[target].copy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print(f'\nTrain/test split: {len(X_train):,} training rows and {len(X_test):,} testing rows')

# IQR OUTLIER HANDLING — bounds are calculated from training data ONLY.
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
outlier_report = []
for column in numeric_features:
    q1, q3 = X_train[column].quantile([0.25, 0.75])
    iqr = q3 - q1
    if pd.isna(iqr) or iqr == 0:
        continue
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    train_outliers = ((X_train[column] < lower) | (X_train[column] > upper)).sum()
    test_outliers = ((X_test[column] < lower) | (X_test[column] > upper)).sum()
    # Cap values rather than deleting rows, preserving the train/test samples.
    X_train[column] = X_train[column].clip(lower, upper)
    X_test[column] = X_test[column].clip(lower, upper)
    outlier_report.append([column, lower, upper, train_outliers, test_outliers])

print('\nIQR OUTLIER REPORT (bounds fit on training data; outliers capped):')
outlier_table = pd.DataFrame(outlier_report, columns=['Feature', 'Lower Bound', 'Upper Bound', 'Train Outliers Capped', 'Test Values Capped']).round(2)
display(outlier_table)

# Nominal categorical fields use one-hot encoding. Ordinal owner fields are converted to an ordered numeric count.
owner_column = next((col for col in X_train.columns if 'owner' in col.lower()), None)
if owner_column:
    owner_map = {'Test Drive Car': 0, 'First Owner': 1, 'Second Owner': 2, 'Third Owner': 3, 'Fourth & Above Owner': 4, 'Fourth And Above Owner': 4}
    X_train[owner_column] = X_train[owner_column].replace(owner_map)
    X_test[owner_column] = X_test[owner_column].replace(owner_map)
    X_train[owner_column] = pd.to_numeric(X_train[owner_column], errors='coerce')
    X_test[owner_column] = pd.to_numeric(X_test[owner_column], errors='coerce')
    print(f'Ordinal encoding applied to: {owner_column}')

numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = [col for col in X_train.columns if col not in numeric_features]
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, numeric_features),
    ('categorical', categorical_pipeline, categorical_features)
])

# fit_transform is deliberately called only for training data; test data is transform-only.
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)
feature_names = preprocessor.get_feature_names_out()
train_processed = pd.DataFrame(X_train_processed, columns=feature_names, index=X_train.index)
test_processed = pd.DataFrame(X_test_processed, columns=feature_names, index=X_test.index)

# Combine transformed features and target into one submission-ready dataset, retaining split information.
train_processed[target] = y_train
train_processed['Dataset_Split'] = 'Train'
test_processed[target] = y_test
test_processed['Dataset_Split'] = 'Test'
processed_dataset = pd.concat([train_processed, test_processed]).sort_index()

print('\nVERIFICATION OF PROCESSED DATASET')
print(f'Processed shape: {processed_dataset.shape[0]:,} rows × {processed_dataset.shape[1]} columns')
print(f'Processed missing values: {processed_dataset.isna().sum().sum():,}')
print(f'Numeric features scaled: {len(numeric_features)}')
print(f'Nominal features one-hot encoded: {len(categorical_features)}')
print(f'Generated model features: {len(feature_names)}')
display(processed_dataset.head())

output_file = 'preprocessed_used_car_resale_dataset.csv'
processed_dataset.to_csv(output_file, index=False)
print(f'\nSaved and downloading: {output_file}')
files.download(output_file)